# 📊 SelvaSonic — Visualización del Dataset

**Notebook 02** — Análisis exploratorio (EDA) del dataset listo para entrenamiento.

## 🎯 ¿Qué hace este notebook?

Es un **diagnóstico crítico** ANTES de entrenar el modelo. Visualiza el dataset construido por `src/dataset.py` y responde preguntas que de otra forma solo descubriríamos cuando el entrenamiento falle:

| Pregunta | Sección que la responde |
|---|---|
| ¿Mis clases están balanceadas? | Sección 2 |
| ¿Mis audios tienen duraciones razonables? | Sección 3 |
| ¿Los espectrogramas se ven bien? | Sección 4 |
| ¿La augmentation hace lo que espero? | Sección 5 |
| ¿El split estratificado por archivo funciona? | Sección 6 |

## 📂 Estructura

0. Imports y configuración
1. Construir el índice del dataset
2. Distribución de clases
3. Distribución de duraciones de audio por clase
4. Mosaico de espectrogramas (1 ejemplo por especie)
5. Comparativa: clip original vs cada augmentation
6. Verificación del split estratificado por archivo

---

**Autoras(es):** Laura Ruiz Arango & Jose Aldair Molina Méndez  
**Universidad:** Universidad Nacional de Colombia — Sede Medellín  
**Materia:** Aprendizaje Automático — Prof. Alcides Montoya

## 0. Imports y configuración

In [ ]:
# ============================================================================
# Imports
# ============================================================================
# El notebook está en notebooks/, los módulos en src/.
# Ajustamos el path para poder importar desde src/.
import sys
from pathlib import Path

# Subir un nivel: notebooks/ -> raíz del proyecto
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Existe data/raw: {(PROJECT_ROOT / 'data' / 'raw').exists()}")

In [ ]:
# Librerías estándar de análisis y visualización
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Módulos del proyecto
from src.audio_io import load_audio
from src.augmentation import add_noise, pitch_shift, time_stretch
from src.config import (
    CLIP_DURATION_S,
    CLIP_NUM_SAMPLES,
    HOP_LENGTH,
    N_FFT,
    N_MELS,
    SAMPLE_RATE,
)
from src.dataset import (
    SelvaSonicDataset,
    build_index,
    records_to_dataframe,
    stratified_split_by_file,
)
from src.transforms import compute_mel_spectrogram, normalize_spectrogram

# Configuración estética de los gráficos
# Usamos seaborn para un estilo limpio + matplotlib para control fino
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["figure.figsize"] = (10, 5)

# Semilla para reproducibilidad de muestreos aleatorios en este notebook
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

print("Imports OK")
print(f"\nConstantes del proyecto:")
print(f"  SAMPLE_RATE      = {SAMPLE_RATE} Hz")
print(f"  CLIP_DURATION_S  = {CLIP_DURATION_S} s")
print(f"  CLIP_NUM_SAMPLES = {CLIP_NUM_SAMPLES}")
print(f"  N_MELS           = {N_MELS}")
print(f"  N_FFT            = {N_FFT}")
print(f"  HOP_LENGTH       = {HOP_LENGTH}")

## 1. Construir el índice del dataset

Llamamos a `build_index()` (módulo `src/dataset.py`) para que recorra `data/raw/`, segmente cada audio y construya la **"libreta de coordenadas"**: una lista de `SegmentRecord` con la metadata de cada clip (sin cargar audio en RAM).

**¿Cuánto tarda?** Cargar y segmentar los ~334 audios de Xeno-canto + ~1600 de ESC-50 toma alrededor de **1-2 minutos**. Es una operación que se hace UNA SOLA VEZ por sesión (después podemos reutilizar el índice).

**Output esperado:** lista de `SegmentRecord` + `label_map` (nombre clase → etiqueta entera).

In [ ]:
# Construir índice maestro (esto puede tardar 1-2 minutos)
raw_dir = PROJECT_ROOT / "data" / "raw"
records, label_map = build_index(raw_dir, verbose=True)

# Convertir a DataFrame para análisis tabular fácil
df = records_to_dataframe(records)

# Mapeo inverso: etiqueta entera -> nombre (útil para gráficos)
label_to_name = {v: k for k, v in label_map.items()}

print(f"\nÍndice construido: {len(df)} clips totales")
print(f"Clases: {len(label_map)}")
df.head()

## 2. Distribución de clases

**¿Por qué importa?** Si una clase tiene 100x más ejemplos que otra (desbalance fuerte), el modelo va a aprender a predecir siempre la clase mayoritaria porque eso le da accuracy alta gratis. Los modelos resultan engañosamente "buenos" en accuracy pero son inútiles en producción.

**¿Qué buscar?**

- Si el balance es razonable (ratio máximo:mínimo < 5x): podemos entrenar tal cual.
- Si hay desbalance fuerte (ratio > 10x): tenemos que aplicar **class weights** en la loss function (planeado para Semana 4).
- En SelvaSonic es **esperado** que `no_ave` sea mucho más grande que cada especie individual (ESC-50 trae ~1600 audios, mientras que cada especie de ave trae 20-50 audios).

In [ ]:
# Contar clips por clase
class_counts = df["label_name"].value_counts().sort_values(ascending=True)

# Calcular ratios para diagnóstico
max_count = class_counts.max()
min_count = class_counts.min()
ratio = max_count / min_count

# Tabla resumen
summary = pd.DataFrame({
    "clase": class_counts.index,
    "clips": class_counts.values,
    "%_total": (100 * class_counts.values / class_counts.sum()).round(2),
})
print("Distribución de clases (de menor a mayor):")
print(summary.to_string(index=False))
print(f"\nRatio max/min = {ratio:.1f}x")
if ratio > 10:
    print("DIAGNÓSTICO: desbalance fuerte (>10x). Usar class_weights en la loss.")
elif ratio > 5:
    print("DIAGNÓSTICO: desbalance moderado. Considerar class_weights.")
else:
    print("DIAGNÓSTICO: dataset razonablemente balanceado.")

In [ ]:
# Gráfico de barras horizontal — más legible cuando hay 11 clases
fig, ax = plt.subplots(figsize=(10, 6))

# Color especial para la clase no_ave (es la negativa, conviene destacarla)
colors = ["#888780" if name == "no_ave" else "#1D9E75"
          for name in class_counts.index]

bars = ax.barh(class_counts.index, class_counts.values, color=colors, edgecolor="white")

# Anotar el conteo al final de cada barra
for bar, count in zip(bars, class_counts.values):
    ax.text(
        bar.get_width() + max_count * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{count}",
        va="center", fontsize=10,
    )

ax.set_xlabel("Número de clips de 5 segundos")
ax.set_title("Distribución de clips por clase", fontsize=13, pad=15)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 3. Distribución de duraciones de audio por clase

**¿Por qué importa?** Necesitamos verificar que los audios tienen duraciones razonables ANTES de entrenar:

- **Audios muy cortos (< 2s):** vienen de grabaciones incidentales o cortes mal hechos. Pueden distorsionar el aprendizaje.
- **Audios muy largos (> 60s):** generan muchos clips del mismo archivo, dominando la clase. Posible sesgo.

**Métrica clave:** la duración **del archivo original**, no del clip. Recordemos que un archivo de 30s genera 6 clips de 5s.

Calculamos la duración de cada archivo a partir del índice (es el `end_sec` máximo por archivo).

In [ ]:
# Para cada archivo único, su duración es el end_sec más alto entre sus clips
file_durations = df.groupby(["file_id", "label_name"])["end_sec"].max().reset_index()
file_durations.rename(columns={"end_sec": "duration_sec"}, inplace=True)

# Estadísticas básicas globales
print("Estadísticas globales de duración (en segundos):")
print(file_durations["duration_sec"].describe().round(2).to_string())
print(f"\nTotal archivos únicos: {len(file_durations)}")

In [ ]:
# Boxplot por clase — muestra mediana, cuartiles y outliers
fig, ax = plt.subplots(figsize=(11, 6))

# Ordenar clases para que aparezcan en orden alfabético + no_ave al final
class_order = [c for c in sorted(file_durations["label_name"].unique())
                if c != "no_ave"] + ["no_ave"]

sns.boxplot(
    data=file_durations,
    x="label_name",
    y="duration_sec",
    order=class_order,
    ax=ax,
    color="#5DCAA5",
    fliersize=3,
)

# Línea horizontal en CLIP_DURATION_S como referencia
ax.axhline(
    y=CLIP_DURATION_S, color="#993C1D", linestyle="--", linewidth=1,
    label=f"CLIP_DURATION_S = {CLIP_DURATION_S}s",
)

ax.set_xlabel("Clase")
ax.set_ylabel("Duración del archivo (segundos)")
ax.set_title("Distribución de duraciones por clase", fontsize=13, pad=15)
ax.tick_params(axis="x", rotation=45)
for label in ax.get_xticklabels():
    label.set_horizontalalignment("right")
ax.legend(loc="upper right")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Histograma agregado: ¿cómo se distribuyen TODAS las duraciones?
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(
    file_durations["duration_sec"],
    bins=40, color="#1D9E75", edgecolor="white", alpha=0.85,
)
ax.axvline(
    x=CLIP_DURATION_S, color="#993C1D", linestyle="--", linewidth=1.5,
    label=f"CLIP_DURATION_S = {CLIP_DURATION_S}s",
)
ax.axvline(
    x=file_durations["duration_sec"].median(), color="#185FA5",
    linestyle=":", linewidth=1.5,
    label=f"mediana = {file_durations['duration_sec'].median():.1f}s",
)

ax.set_xlabel("Duración del archivo (segundos)")
ax.set_ylabel("Número de archivos")
ax.set_title("Histograma global de duraciones", fontsize=13, pad=15)
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 4. Mosaico de espectrogramas (1 muestra por clase)

**¿Por qué importa?** Los espectrogramas Mel son lo que la CNN va a "ver". Antes de entrenar, queremos verificar visualmente:

- **Cada especie tiene una firma espectral distinta?** Si todas se ven iguales, el modelo no podrá distinguirlas.
- **¿Hay artefactos raros?** (líneas verticales = clicks, bandas horizontales = ruido eléctrico).
- **¿La normalización funciona?** Los valores deberían estar centrados (mean ≈ 0, std ≈ 1).

**Interpretación:** los ejes son tiempo (horizontal) y frecuencia (vertical, escala Mel). Los colores brillantes son frecuencias activas en ese momento. Las **firmas espectrales** características de cada ave aparecen como patrones repetidos.

In [ ]:
# Seleccionar 1 clip aleatorio por clase usando el índice
samples_per_class = []
for label, name in sorted(label_to_name.items()):
    candidates = [r for r in records if r.label == label and not r.is_padded]
    if not candidates:
        candidates = [r for r in records if r.label == label]
    sample_idx = rng.integers(0, len(candidates))
    samples_per_class.append((name, candidates[sample_idx]))

print(f"Seleccionadas {len(samples_per_class)} muestras (1 por clase)")

In [ ]:
import librosa.display

# Calcular layout del mosaico (3-4 columnas según número de clases)
n = len(samples_per_class)
n_cols = 3
n_rows = int(np.ceil(n / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3.5 * n_rows))
axes = axes.flatten()

for ax, (class_name, rec) in zip(axes, samples_per_class):
    # Cargar SOLO el segmento del clip (offset + duration)
    duration = rec.end_sec - rec.start_sec
    audio = load_audio(
        rec.file_path, target_sr=SAMPLE_RATE,
        offset=rec.start_sec, duration=duration,
    )

    # Calcular Mel-spectrograma usando funciones del proyecto
    mel_db = compute_mel_spectrogram(audio.waveform, sr=SAMPLE_RATE)

    # Visualizar con librosa.display.specshow (maneja ejes correctamente)
    img = librosa.display.specshow(
        mel_db, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
        x_axis="time", y_axis="mel", fmin=50, fmax=11025,
        ax=ax, cmap="magma",
    )
    ax.set_title(f"{class_name}\n({Path(rec.file_path).name})", fontsize=10)
    ax.set_xlabel("")
    ax.set_ylabel("")

# Apagar ejes sobrantes
for ax in axes[n:]:
    ax.axis("off")

fig.suptitle(
    "Mel-spectrogramas (1 muestra por clase) — escala dB, eje Mel",
    fontsize=13, y=1.00,
)
fig.colorbar(img, ax=axes[:n], format="%+2.0f dB", shrink=0.6, pad=0.02)
plt.tight_layout()
plt.show()

## 5. Comparativa: clip original vs cada augmentation

**¿Por qué importa?** Las augmentations modifican el audio para crear variaciones sintéticas que ayudan al modelo a generalizar. Antes de confiar en ellas, queremos verificar VISUALMENTE que cada una hace lo correcto:

| Augmentation | Qué debería verse en el espectrograma |
|---|---|
| **time_stretch** | Mismas alturas (frecuencias), pero comprimido o expandido en tiempo |
| **pitch_shift** | Misma duración, pero las firmas espectrales SUBEN o BAJAN en el eje frecuencia |
| **add_noise** | Misma estructura, pero con "granulado" (ruido) sumado a todo |

Si en algún caso vemos algo diferente, hay un bug en la implementación.

In [ ]:
# Tomamos un clip de ave (NO no_ave) para que las augmentations sean más visibles
ave_records = [r for r in records if r.label != 0 and not r.is_padded]
demo_idx = rng.integers(0, len(ave_records))
demo_rec = ave_records[demo_idx]

print(f"Clip seleccionado para demo:")
print(f"  Especie: {demo_rec.label_name}")
print(f"  Archivo: {Path(demo_rec.file_path).name}")
print(f"  Segmento: [{demo_rec.start_sec:.2f}s - {demo_rec.end_sec:.2f}s]")

# Cargar el clip original (sin augmentation)
duration = demo_rec.end_sec - demo_rec.start_sec
demo_audio = load_audio(
    demo_rec.file_path, target_sr=SAMPLE_RATE,
    offset=demo_rec.start_sec, duration=duration,
)
demo_samples = demo_audio.waveform

# Asegurar longitud exacta (por si tiene padding)
if demo_samples.shape[0] < CLIP_NUM_SAMPLES:
    reps = int(np.ceil(CLIP_NUM_SAMPLES / demo_samples.shape[0]))
    demo_samples = np.tile(demo_samples, reps)[:CLIP_NUM_SAMPLES]
else:
    demo_samples = demo_samples[:CLIP_NUM_SAMPLES]

print(f"  Shape del clip: {demo_samples.shape}")

In [ ]:
# Generar las 3 augmentations con seed fija para reproducibilidad de la demo
DEMO_SEED = 123

# 1) Time stretch con factor explícito (más rápido) para que el efecto se vea claro
stretched = time_stretch(
    demo_samples, sample_rate=SAMPLE_RATE, rate=1.2, random_state=DEMO_SEED,
)

# 2) Pitch shift +2 semitones (más agudo)
shifted = pitch_shift(
    demo_samples, sample_rate=SAMPLE_RATE, n_semitones=2, random_state=DEMO_SEED,
)

# 3) Add noise con SNR=15dB (ruido moderado-fuerte para que se note)
noisy = add_noise(demo_samples, snr_db=15.0, random_state=DEMO_SEED)

# Calcular Mel-spectrogramas de los 4 (original + 3 augmentations)
mels = {
    "Original": compute_mel_spectrogram(demo_samples, sr=SAMPLE_RATE),
    "Time stretch (rate=1.2)": compute_mel_spectrogram(stretched, sr=SAMPLE_RATE),
    "Pitch shift (+2 semitones)": compute_mel_spectrogram(shifted, sr=SAMPLE_RATE),
    "Add noise (SNR=15 dB)": compute_mel_spectrogram(noisy, sr=SAMPLE_RATE),
}

print("Augmentations generadas correctamente")
for name, m in mels.items():
    print(f"  {name}: shape={m.shape}, range=[{m.min():.1f}, {m.max():.1f}] dB")

In [ ]:
# Mosaico 2x2 comparando original vs cada augmentation
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for ax, (title, mel_db) in zip(axes, mels.items()):
    img = librosa.display.specshow(
        mel_db, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
        x_axis="time", y_axis="mel", fmin=50, fmax=11025,
        ax=ax, cmap="magma",
    )
    ax.set_title(title, fontsize=11)

fig.suptitle(
    f"Augmentation comparativa — clip de {demo_rec.label_name}",
    fontsize=13, y=0.995,
)
fig.colorbar(img, ax=axes, format="%+2.0f dB", shrink=0.6, pad=0.02)
plt.tight_layout()
plt.show()

### 🔍 Análisis de la comparativa

**Comparando con el original**:

1. **Time stretch (rate=1.2):** notar que el patrón llega "antes" en el tiempo (audio acelerado). Las frecuencias (eje Y) se mantienen iguales. ✅

2. **Pitch shift (+2 semitones):** notar que las firmas espectrales aparecen DESPLAZADAS HACIA ARRIBA en el eje Y. Las frecuencias se multiplicaron por $2^{2/12} \approx 1.122$. La duración se mantiene. ✅

3. **Add noise (SNR=15dB):** notar el "granulado" o "niebla" uniforme sumado al espectrograma. Las firmas del canto siguen siendo visibles, pero hay energía de fondo en todas las frecuencias y tiempos. ✅

Si visualmente todo cuadra → la augmentation está bien implementada y podemos usarla con confianza en entrenamiento.

## 6. Verificación del split estratificado por archivo

**¿Por qué importa?** Esta es la regla de oro de la metodología:

1. **Estratificación:** todas las clases deben aparecer en los 3 splits con proporciones similares.
2. **Integridad por archivo:** ningún `file_id` debe aparecer en dos splits distintos (evita data leakage por ambiente compartido).

Vamos a verificar AMBAS condiciones gráficamente.

In [ ]:
# Aplicar el split estratificado por archivo
train_recs, val_recs, test_recs = stratified_split_by_file(
    records,
    train_ratio=0.70, val_ratio=0.15, test_ratio=0.15,
    random_state=RANDOM_STATE,
)

print(f"Split aplicado:")
print(f"  Train: {len(train_recs):>5} clips")
print(f"  Val:   {len(val_recs):>5} clips")
print(f"  Test:  {len(test_recs):>5} clips")
print(f"  Total: {len(train_recs)+len(val_recs)+len(test_recs):>5} clips")

In [ ]:
# Verificación 1: integridad por archivo (ningún file_id en dos splits)
train_files = {r.file_id for r in train_recs}
val_files = {r.file_id for r in val_recs}
test_files = {r.file_id for r in test_recs}

leak_tv = train_files & val_files
leak_tt = train_files & test_files
leak_vt = val_files & test_files

print("Verificación de integridad por archivo (regla de oro):")
print(f"  Train ∩ Val:  {len(leak_tv)} archivos compartidos")
print(f"  Train ∩ Test: {len(leak_tt)} archivos compartidos")
print(f"  Val ∩ Test:   {len(leak_vt)} archivos compartidos")

if not (leak_tv or leak_tt or leak_vt):
    print("\nPASS: ningún data leakage por archivo entre splits")
else:
    print("\nFAIL: hay archivos en múltiples splits, revisar split function")

In [ ]:
# Verificación 2: estratificación (proporciones por clase en cada split)
df_train = records_to_dataframe(train_recs)
df_val = records_to_dataframe(val_recs)
df_test = records_to_dataframe(test_recs)

# Contar clips por (clase, split)
split_counts = pd.DataFrame({
    "train": df_train["label_name"].value_counts(),
    "val": df_val["label_name"].value_counts(),
    "test": df_test["label_name"].value_counts(),
}).fillna(0).astype(int)

# Ordenar como en gráficos previos
split_counts = split_counts.reindex(class_order)

print("Tabla de clips por clase y split:")
print(split_counts.to_string())

# Calcular proporciones (cada fila debe sumar 100%)
split_pct = split_counts.div(split_counts.sum(axis=1), axis=0) * 100
print("\nProporción por split (cada fila suma 100%):")
print(split_pct.round(1).to_string())

In [ ]:
# Gráfico de barras agrupadas: 1 grupo por clase, 3 barras (train/val/test)
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(split_counts))
width = 0.27

ax.bar(x - width, split_counts["train"], width, label="Train", color="#1D9E75")
ax.bar(x, split_counts["val"], width, label="Val", color="#EF9F27")
ax.bar(x + width, split_counts["test"], width, label="Test", color="#D85A30")

ax.set_xticks(x)
ax.set_xticklabels(split_counts.index, rotation=45, ha="right")
ax.set_ylabel("Número de clips")
ax.set_title("Distribución de clips por clase y split", fontsize=13, pad=15)
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Stacked bar al 100% — visualiza si las proporciones train/val/test
# son consistentes entre clases (deberían serlo por la estratificación)
fig, ax = plt.subplots(figsize=(12, 6))

split_pct.plot(
    kind="bar", stacked=True, ax=ax,
    color=["#1D9E75", "#EF9F27", "#D85A30"], width=0.7, edgecolor="white",
)

# Líneas horizontales en 70% y 85% como referencia (split objetivo)
ax.axhline(y=70, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
ax.axhline(y=85, color="black", linestyle="--", linewidth=0.8, alpha=0.5)

ax.set_ylim(0, 100)
ax.set_ylabel("% de clips de la clase")
ax.set_xlabel("Clase")
ax.set_title(
    "Proporción train/val/test por clase (objetivo: 70/15/15)",
    fontsize=13, pad=15,
)
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
ax.tick_params(axis="x", rotation=45)
for label in ax.get_xticklabels():
    label.set_horizontalalignment("right")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## ✅ Resumen del diagnóstico

Si todas las celdas anteriores se ejecutaron correctamente, hemos verificado:

1. **Sección 2 — Distribución de clases:** sabemos si necesitamos `class_weights` en la loss (Semana 4).
2. **Sección 3 — Duraciones:** confirmamos que los audios tienen duraciones razonables.
3. **Sección 4 — Espectrogramas:** verificamos visualmente que cada clase tiene firma espectral distintiva.
4. **Sección 5 — Augmentation:** confirmamos que las 3 augmentations hacen lo que esperamos.
5. **Sección 6 — Split estratificado:** validamos que no hay data leakage entre splits y las proporciones son correctas.

**El pipeline de datos de SelvaSonic está listo para alimentar al modelo en Semana 3.** 🚀

## 📝 Próximos pasos (Semana 3)

- Diseñar arquitectura CNN (`src/model.py`)
- Implementar training loop (`src/train.py`)
- Primer entrenamiento baseline
- Análisis de errores